In [ ]:
# --- Hàm xây dựng Ma trận A (từ Ảnh 2) ---
def build_A_matrix(n_goods, n_slots):
    """Tạo ma trận ánh xạ Goods -> Slots"""
    A = np.zeros((n_slots, n_goods))
    for j in range(n_goods):
        A[j % n_slots, j] = 1.0
    return A

In [ ]:
# --- Hàm buyer best response for each user used in SPDS
def buyer_best_response_cvx(v_i, p, A, q_i, B_i, utility_type="Linear"):
    """
    Phiên bản siêu tốc của buyer_best_response.
    Sử dụng Closed-form solution cho mọi hàm utility.
    KHÔNG DÙNG CVXPY.
    """
    # 1. Tính giá hiệu dụng (Effective Price)
    # p: (M,), A.T @ q_i: (M,)
    effective_price = p + (A.T @ q_i)

    # An toàn: Đảm bảo giá > 0 để tránh chia cho 0
    safe_price = np.maximum(effective_price, 1e-9)

    m_goods = len(v_i)
    x = np.zeros(m_goods)

    # =========================================================
    # 1. LINEAR UTILITY: U = sum(v * x)
    # =========================================================
    if utility_type == "Linear":
        # Chiến thuật: Bang-per-buck (Mua tất tay món hời nhất)
        bang_per_buck = v_i / safe_price
        best_idx = np.argmax(bang_per_buck)
        x[best_idx] = B_i / safe_price[best_idx]

    # =========================================================
    # 2. COBB-DOUGLAS: U = prod(x^alpha) hoặc sum(alpha * log(x))
    # =========================================================
    elif utility_type == "Cobb-Douglas":
        # Chuẩn hóa alpha (Bắt buộc để thỏa mãn Budget constraint)
        sum_v = np.sum(v_i)
        if sum_v > 0:
            alpha = v_i / sum_v
        else:
            alpha = np.zeros_like(v_i) # Tránh lỗi nếu v toàn 0

        # Công thức: Chi tiêu đúng tỷ lệ alpha
        # x_i = (alpha_i * Budget) / Price_i
        x = (alpha * B_i) / safe_price

    # =========================================================
    # 3. LEONTIEF: U = min(x / v)
    # =========================================================
    elif utility_type == "Leontief":
        # Chiến thuật: Mua theo tỷ lệ cố định của v
        # Giá của 1 combo chuẩn = sum(v_i * p_i)
        cost_of_one_bundle = np.dot(v_i, safe_price)

        if cost_of_one_bundle > 0:
            # Số lượng combo mua được
            num_bundles = B_i / cost_of_one_bundle
            x = num_bundles * v_i
        else:
            x = np.zeros(m_goods)

    # =========================================================
    # 4. CES UTILITY: U = (sum v * x^rho)^(1/rho)
    # =========================================================
    elif utility_type == "CES_8":
        # CẤU HÌNH rho (m) TẠI ĐÂY
        # Trong code cũ bạn để m = 1/2
        rho = 0.5

        # --- CASE A: CONCAVE (rho < 1, rho != 0) ---
        # Đây là trường hợp thay thế (Substitute) -> Mua nhiều loại
        if rho < 1:
            # Tính Sigma (Elasticity of Substitution)
            # sigma = 1 / (1 - rho)
            sigma = 1.0 / (1.0 - rho)

            # Tính phần tử tỷ lệ: Term_i = (v_i / p_i)^sigma
            # Dùng np.maximum cho v_i để tránh v=0 gây lỗi log hoặc mũ âm
            term = np.power(v_i / safe_price, sigma)

            # Tính mẫu số chung: Sum (p_j * term_j)
            denom = np.dot(safe_price, term)

            if denom > 0:
                # x_i = (B * term_i) / denom
                x = (B_i * term) / denom
            else:
                 # Fallback nếu v=0 hết
                 x = np.zeros(m_goods)

        # --- CASE B: CONVEX (rho > 1) ---
        # Đây là trường hợp "Winner Takes All" giống Linear
        else:
            # So sánh tỷ lệ: v^(1/rho) / p
            v_transformed = np.power(v_i, 1.0/rho)
            bang_per_buck = v_transformed / safe_price

            best_idx = np.argmax(bang_per_buck)
            x[best_idx] = B_i / safe_price[best_idx]

    return x

# --- GROUND TRUTH OPTIMAL RESULT
def solve_centralized_optimal_fair(valuations, budgets, supply_s, b_mat, A, utility_type="Linear"):
    """
    Returns:
        optimal_value (float): Giá trị hàm mục tiêu tối ưu.
        optimal_X (np.ndarray): Ma trận phân bổ tối ưu (N x M).
    """
    n, m = valuations.shape
    X = cp.Variable((n, m), nonneg=True)

    constraints = [
        cp.sum(X, axis=0) <= supply_s
    ]
    for i in range(n):
        constraints.append(A @ X[i] <= b_mat[i])

    # --- XÂY DỰNG OBJECTIVE ---
    if utility_type == "Linear":
        utilities = cp.sum(cp.multiply(valuations, X), axis=1)
        primal_utility = cp.sum(cp.multiply(budgets, cp.log(utilities + 1e-12)))

    elif utility_type == "Cobb-Douglas":
        eps = 1e-12
        alpha = valuations / (np.sum(valuations, axis=1, keepdims=True))
        log_utilities = cp.sum(cp.multiply(alpha, cp.log(X + eps)), axis=1)
        primal_utility = cp.sum(cp.multiply(budgets, log_utilities))

    elif utility_type == "Leontief":
        # Lưu ý: Leontief trong CVXPY có thể phức tạp/chậm với quy mô lớn
        inv_valuations = np.divide(
            1.0,
            valuations,
            out=np.full_like(valuations, 1e30),
            where=(valuations > 0)
        )

        # 2. Nhân X với ma trận nghịch đảo hằng số này
        # ratio_matrix[i, j] = X[i, j] * (1 / v[i, j])
        weighted_X = cp.multiply(X, inv_valuations)

        # 3. Lấy min theo hàng
        utilities = cp.min(weighted_X, axis=1)

        # 4. Tính hàm mục tiêu
        primal_utility = cp.sum(cp.multiply(budgets, cp.log(utilities + 1e-12)))

    elif utility_type == "CES_8":
        # Nếu chạy Solver tập trung, m NÊN nhỏ hơn hoặc bằng 1 (ví dụ 0.5 hoặc -1).
        # Nếu đặt m = 8, Solver ECOS/SCS sẽ KHÔNG giải được (báo lỗi DCPError).
        eps = 1e-9
        m_ces = 1/2

        # Công thức: log_util = (1/m) * log( sum( v * x^m ) )
        # Tính tổng trọng số lũy thừa theo hàng (axis=1)
        inner_term = cp.sum(cp.multiply(valuations, cp.power(X + eps, m_ces)), axis=1)

        # Logarit hóa hàm mục tiêu
        log_utilities = (1.0 / m_ces) * cp.log(inner_term)

        primal_utility = cp.sum(cp.multiply(budgets, log_utilities))

    objective = cp.Maximize(primal_utility)
    prob = cp.Problem(objective, constraints)

    print(f"--- Solving Centralized Fair Problem ({utility_type}) ---")
    try:
        prob.solve(solver=cp.ECOS, verbose=False)
    except:
        prob.solve(solver=cp.SCS, verbose=False)

    # TRẢ VỀ: (Giá trị mục tiêu, Ma trận X tối ưu)
    return prob.value, X.value

In [ ]:
def full_grad_descent(
    supply_s, capacity_b, A, valuations, budgets,
    p0, q0, lr_p=0.02, lr_q=0.02, num_iters=1000000,
    log_freq=250, seed=42, eps=0.00001):
    """
    Full primal-dual gradient descent with capacity constraints
    """

    np.random.seed(seed)

    n, m = valuations.shape

    p = p0.astype(float).copy()          # goods prices (m,)
    q = q0.astype(float).copy()          # capacity prices (n,T)

    # --- BƯỚC 1: KHỞI TẠO TUYỆT ĐỐI (X0 = 100) ---
    X = np.ones((n, m)) * 1
    util =  np.zeros(n)
    for j in range(n):
        util[j] = valuations[j] @ X[j]

    # --- 2. GHI LOG TRẠNG THÁI BAN ĐẦU (t=0) ---
    # Khởi tạo lịch sử
    num_logs = num_iters // log_freq + 1
    obj_hist = np.empty(num_logs)
    time_hist = np.empty(num_logs)
    log_idx = 0

    # Tính Objective tại điểm xuất phát (lúc này chưa ai mua gì)
    term_p = np.sum(p * supply_s)     # p * Supply
    term_q = np.sum(q * capacity_b)            # q * Capacity
    term_u = np.sum(budgets * np.log(util + 1e-12)) # Utility
    obj_0 = term_p + term_q + term_u - np.sum(budgets)

    obj_hist[log_idx] = obj_0
    time_hist[log_idx] = 0.0
    log_idx += 1

    print(f"--- Start GD ({num_iters} iterations) | Initial Obj: {obj_0:.2f} ---")

    # Easier for final allocation
    X = np.array([buyer_best_response_cvx(valuations[j], p, A, q[j], budgets[j])
                for j in range(n)])


    try:
        for t in range(1, num_iters + 1):
            elapsed = time.perf_counter() - t0

            # ---------------------------
            # Buyer's best response
            # ---------------------------
            X = np.array([buyer_best_response_cvx(valuations[j], p, A, q[j], budgets[j])
                for j in range(n)])
            util = np.sum(valuations * X, axis=1)

            # ---------------------------
            # Dual gradients
            # ---------------------------
            g_p = X.sum(axis=0) - supply_s   # (1, m)
            g_q = (A @ X.T).T - capacity_b  # (n,T)

            # Step sizes
            alpha = lr_p / np.sqrt(t)
            beta  = lr_q / np.sqrt(t)

            # Dual updates
            p = np.maximum(p + alpha * g_p, 0.0)
            q = np.maximum(q + beta  * g_q, 0.0)

            term_u = np.sum(budgets * np.log(util + 1e-12))
            term_p = np.sum(p * supply_s)
            term_q = np.sum(q * capacity_b)
            obj = term_u + term_p + term_q - np.sum(budgets)

            # ---------------------------
            # Logging
            # ---------------------------
            if t % log_freq == 0:
                # Update history
                obj_hist[log_idx] = obj
                time_hist[log_idx] = elapsed

                # Update log_idx in numpy array
                log_idx += 1

                # ---------- Termination conditions ----------
                ratio = abs((obj_hist[log_idx - 1] - obj_hist[log_idx - 2]) / obj_hist[log_idx - 2])
                if ratio <= eps:
                    print(f"DPDS Iterations: {t}")
                    print(f"Objective value: {obj:.4f}")
                    print(f"DPDS Algorithm finsished at epsilon difference {eps} in {elapsed:.2f} s")
                    break

    except KeyboardInterrupt:
        print("🛑 Interrupted by user")

    except Exception as e:
        print(f"💥 Crash: {e}")

    return obj_hist[:log_idx], time_hist[:log_idx], X.copy(), p, q